In [ ]:
!nvidia-smi

Fri Apr 24 07:10:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
%%writefile vector_addition.cu
#include <stdio.h>
#include <cuda_runtime.h>

#define N 8

// GPU kernel: each thread adds one pair of elements
__global__ void vectorAdd(int *A, int *B, int *C) {
    int i = threadIdx.x;   // thread index = array index
    C[i] = A[i] + B[i];
}

int main() {
    int h_A[N] = {1, 2, 3, 4, 5, 6, 7, 8};
    int h_B[N] = {10, 20, 30, 40, 50, 60, 70, 80};
    int h_C[N];

    int *d_A, *d_B, *d_C;
    int size = N * sizeof(int);

    // Step 1: Allocate memory on GPU
    cudaMalloc(&d_A, size);
    cudaMalloc(&d_B, size);
    cudaMalloc(&d_C, size);

    // Step 2: Copy data from CPU to GPU
    cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);

    // Step 3: Launch kernel — 1 block, N threads
    vectorAdd<<<1, N>>>(d_A, d_B, d_C);

    // Step 4: Copy result from GPU back to CPU
    cudaMemcpy(h_C, d_C, size, cudaMemcpyDeviceToHost);

    printf("A:      ");
    for (int i = 0; i < N; i++) printf("%3d ", h_A[i]);
    printf("\nB:      ");
    for (int i = 0; i < N; i++) printf("%3d ", h_B[i]);
    printf("\nA + B:  ");
    for (int i = 0; i < N; i++) printf("%3d ", h_C[i]);
    printf("\n");

    // Step 5: Free GPU memory
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}

Writing vector_addition.cu


In [ ]:
# Compile
!nvcc vector_addition.cu -o vector_addition
print('Compiled successfully!')

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Compiled successfully!


In [ ]:
# Run
!./vector_addition

A:        1   2   3   4   5   6   7   8 
B:       10  20  30  40  50  60  70  80 
A + B:   11  22  33  44  55  66  77  88 
